# Talking Medicine — GPT-4o Implementation Notebook

This notebook implements the technical workflow for the *Talking Medicine* framework.

It supports two input workflows:

1. **Manual input workflow**: paste a medical text manually and simplify it for a selected patient profile.
2. **Med-EASi dataset workflow**: iterate over selected examples from the Med-EASi dataset and simplify them in batches.

The notebook uses a two-part prompt architecture:

- **Profile-specific prompt design**: describes the target reader and checklist-based instructions, without exposing complexity-dimension labels in the prompt text.
- **Shared prompt wrapper**: adds the task, target output language, medical correctness constraints, source text, and output requirements.

The patient-profile prompt designs are intentionally language-independent. The output language is specified at runtime in the shared prompt wrapper.

## 1. Install Dependencies

Run this cell once in Colab or a fresh local environment. If you already have the packages installed, you can skip it.

In [2]:
# Uncomment if needed in Colab or a fresh environment
! pip install openai python-dotenv pandas datasets tqdm bert-score

  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
Using cached bert_score-0.3.13-py3-none-any.whl (61 kB)


## 2. Imports and Global Settings

In [3]:
from __future__ import annotations

import os
import json
import time
import uuid
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Any
import bert_score

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

try:
    from openai import AzureOpenAI
except ImportError:
    AzureOpenAI = None
    print("openai package is not installed. Run the install cell first.")

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 10.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 18.7 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -------------- ------------------------- 3.7/9.9 MB 17.6 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.9 MB 19.3 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 15.6 MB/s  0:00:00
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 529.0/529.0 kB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 663.6/663.6 kB 18.0 MB/s  0:00:00
   ---------------------------------------

c:\Users\timos\Desktop\Bachelorarbeit\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Safety switch: keep False while testing notebook structure.
# Set to True only when your Azure OpenAI credentials are configured.
RUN_API_CALLS = True

# Default target language. Change this at runtime if needed, e.g. "German", "English", "French".
TARGET_LANGUAGE = "English"

# Default model settings
TEMPERATURE = 0.2
MAX_TOKENS = 1600

# Folder structure
BASE_DIR = Path.cwd()
PROMPT_DIR = BASE_DIR / "prompts"
OUTPUT_DIR = BASE_DIR / "outputs"
RESULTS_DIR = OUTPUT_DIR / "generations"
EVAL_DIR = OUTPUT_DIR / "evaluations"

for directory in [PROMPT_DIR, OUTPUT_DIR, RESULTS_DIR, EVAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Prompt directory:", PROMPT_DIR)
print("Output directory:", OUTPUT_DIR)

Base directory: c:\Users\timos\Desktop\Bachelorarbeit\Code
Prompt directory: c:\Users\timos\Desktop\Bachelorarbeit\Code\prompts
Output directory: c:\Users\timos\Desktop\Bachelorarbeit\Code\outputs


## 3. Environment Variables and Azure OpenAI Client

Create a `.env` file in the same folder as this notebook, or use Colab Secrets.

Expected variables:

```text
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-api-key
AZURE_OPENAI_API_VERSION=2024-02-15-preview
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=your-gpt-4o-deployment-name
```

The deployment name is the name you gave the model deployment in Azure AI Foundry / Azure OpenAI, not necessarily the base model name.

In [ ]:
# Manual Workaround if you have trouble with .env

AZURE_OPENAI_ENDPOINT = "https://INSERT_YOUR_ENDPOINT.openai.azure.com/"
AZURE_OPENAI_API_KEY = "INSERT_YOUR_KEY"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = "gpt-4o"

print("Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Deployment:", AZURE_OPENAI_CHAT_DEPLOYMENT_NAME)
print("API version:", AZURE_OPENAI_API_VERSION)
print("Key:", AZURE_OPENAI_API_KEY)


Endpoint: https://ai-foundry-bachelorthesis.openai.azure.com/
Deployment: gpt-4o
API version: 2024-12-01-preview
Key: 8QUOlXIw0IIFoVCXWDwp5zuCqwrIaMGmzE6N8fa3WNGeQyre3a9OJQQJ99CEACfhMk5XJ3w3AAAAACOGM1F0


In [6]:
def get_azure_client() -> AzureOpenAI:
    """Create an Azure OpenAI client from environment variables."""
    if AzureOpenAI is None:
        raise ImportError("openai package is not installed.")
    if not AZURE_OPENAI_ENDPOINT or not AZURE_OPENAI_API_KEY:
        raise ValueError("Azure OpenAI endpoint/key missing. Check your .env or Colab Secrets.")
    return AzureOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
    )

client = None
if RUN_API_CALLS:
    client = get_azure_client()
    print("Azure OpenAI client initialized.")
else:
    print("RUN_API_CALLS is False, so the Azure client was not initialized.")

Azure OpenAI client initialized.


## 4. Patient-Profile Prompt Designs

The following prompt designs are based on the current thesis structure:

- patient-profile-specific needs,
- checklist-based operationalization,
- prompt-oriented instructions derived from the framework.

Important: the prompts below do **not** contain the output language and do **not** explicitly list complexity dimensions. The output language is specified in the shared runtime wrapper. The complexity-dimension labels remain part of the framework and evaluation logic, but the generation prompt uses only concrete, patient-oriented instructions.


In [7]:
PROFILE_PROMPT_DESIGNS: Dict[str, str] = {
    "layperson": """
Target patient profile:
The reader is a layperson without specialized medical knowledge. The main barriers are unfamiliar medical terminology, missing background knowledge, and unclear practical implications. They do not need overly-simplified information.

Checklist-based instructions:
- Identify medical terms, acronyms, and abbreviations that may be difficult for the reader.
- Explain necessary medical terms in familiar language.
- Replace unnecessary jargon with simpler expressions where possible.
- Add short background explanations for important medical concepts when the source text assumes medical knowledge.
- Make important relationships between symptoms, risks, diagnoses, or treatments explicit.
- Clarify what the information means for the patient’s own situation.
- Make possible options, consequences, warnings, and next steps explicit when they are present in the source text.
""".strip(),

    "non_native_speaker": """
Target patient profile:
The reader is a non-native speaker. The reader may understand general information but may struggle with technical vocabulary, complex sentence structures, idiomatic wording, and context-dependent expressions. Use plain language.

Checklist-based instructions:
- Use clear and familiar vocabulary.
- Explain necessary medical terms in simple language.
- Clarify abbreviations and acronyms when they may be misunderstood.
- Avoid idioms, metaphors, and culturally specific expressions.
- Split long or nested sentences into shorter sentences.
- Prefer active voice where it makes the meaning clearer.
- Separate different ideas, conditions, or exceptions into distinct sentences.
- Identify words or phrases that could be interpreted in more than one way.
- Clarify terms whose medical meaning differs from their everyday meaning.
- Make implied information explicit where misunderstanding is likely.
""".strip(),

    "cognitive_disabilities": """
Target patient profile:
The reader has cognitive disabilities, learning difficulties, or reduced processing capacity. The reader may be overwhelmed by dense information, complex grammar, implied meaning, unfamiliar terms, or several ideas presented at once. Use a higher grade of simplification and easy-to-read language.

Checklist-based instructions:
- Present one main idea at a time.
- Split long or nested sentences into shorter sentences.
- Separate different topics into clear sections or short paragraphs.
- Use step-by-step structure or lists where they improve orientation.
- Group related information together.
- Explain necessary medical terms in familiar language.
- Clarify abbreviations and acronyms when they may be misunderstood.
- Make implied information explicit where misunderstanding is likely.
- Avoid unnecessary abstraction, idioms, metaphors, and complicated sentence structures.
""".strip(),

    "older_adults": """
Target patient profile:
The reader is an older adult patient. The reader may be affected by dense information, long sentence structures, unclear next steps, unfamiliar medical terms, or ambiguous expressions. The explanation should support careful reading and practical understanding.

Checklist-based instructions:
- Use shorter and more direct sentences.
- Split long or nested sentences into shorter sentences.
- Separate different ideas, conditions, or exceptions into distinct sentences.
- Group related information together.
- Separate different topics into clear paragraphs or sections.
- Use headings or bullet points where they improve orientation.
- Explain important medical terms in familiar language.
- Clarify ambiguous expressions when they could be misunderstood.
- Clarify what the information means for the patient’s own situation.
- State warnings and next steps clearly when they are present in the source text.
- Indicate which points should be discussed with a healthcare professional when this is present or clearly implied in the source text.
""".strip(),

    "lower_education_level": """
Target patient profile:
The reader has a lower educational background. The reader may have had less exposure to academic, scientific, or medical language. Technical terms, abstract concepts, complex sentence structures, and unclear decision implications may create barriers to understanding. Use plain language when explaining.

Checklist-based instructions:
- Use accessible vocabulary.
- Identify medical terms that may be difficult for the reader.
- Explain necessary medical terms in familiar language.
- Replace unnecessary jargon with simpler expressions where possible.
- Add short background explanations for important medical concepts.
- Explain concepts step by step.
- Make important medical relationships and mechanisms explicit.
- Use examples or comparisons where they support understanding.
- Use shorter and more direct sentences.
- Avoid academic or bureaucratic sentence structures.
- Clarify options, consequences, warnings, and next steps when they are present in the source text.
""".strip(),

    # Optional baseline. This is not one of the five patient profiles from the framework.
    "general_non_expert_baseline": """
Target reader:
The reader is a general non-expert without specialized medical training. This prompt is used as a baseline and is not tied to one specific vulnerable patient profile.

Checklist-based instructions:
- Explain necessary medical terms in familiar language.
- Replace unnecessary jargon with simpler expressions where possible.
- Use clear and direct sentence structure.
- Add short background explanations when the source text assumes medical knowledge.
- Clarify wording that could be misunderstood.
- Group related information together.
- Make practical relevance clear when it is present in the source text.
""".strip(),
}



In [8]:
def write_profile_prompt_files(overwrite: bool = True) -> None:
    """Write profile prompt designs to the prompts/ folder."""
    PROMPT_DIR.mkdir(parents=True, exist_ok=True)
    for profile_name, prompt_text in PROFILE_PROMPT_DESIGNS.items():
        path = PROMPT_DIR / f"{profile_name}.txt"
        if path.exists() and not overwrite:
            continue
        path.write_text(prompt_text.strip() + "\n", encoding="utf-8")
    print(f"Wrote {len(PROFILE_PROMPT_DESIGNS)} prompt files to {PROMPT_DIR}")

write_profile_prompt_files(overwrite=True)
print("Available prompt files:")
for p in sorted(PROMPT_DIR.glob("*.txt")):
    print("-", p.name)

Wrote 6 prompt files to c:\Users\timos\Desktop\Bachelorarbeit\Code\prompts
Available prompt files:
- cognitive_disabilities.txt
- general_non_expert_baseline.txt
- layperson.txt
- lower_education_level.txt
- non_native_speaker.txt
- older_adults.txt


## 5. Shared Prompt Wrapper

The shared wrapper adds the parts that remain constant across patient profiles:

- task definition,
- output language,
- medical correctness constraints,
- output requirements,
- source text insertion.

The profile-specific prompt design is inserted into `{profile_prompt}` and the original text is inserted into `{source_text}`.

In [ ]:
SYSTEM_MESSAGE = """
You are a careful medical text simplification assistant. Your task is to rewrite medical information for patients while preserving the meaning of the source text. 
You must follow the provided patient-profile prompt design and the medical correctness constraints exactly.
""".strip()

SHARED_PROMPT_WRAPPER = """
Task:
Rewrite the source text into a patient-facing simplified version.

Output language:
{target_language}

Use the patient-profile prompt design below to guide the simplification.

Patient-profile prompt design:
{profile_prompt}

Medical correctness constraints:
- Preserve the medical meaning of the source text.
- Do not invent diagnoses, symptoms, risks, treatments, warnings, or recommendations that are not supported by the source text.
- Do not remove medically relevant information.
- Preserve uncertainty, conditions, warnings, and risk statements.
- Do not make uncertain information sound certain.
- You may add short explanatory background only when it helps explain a medical concept already present in the source text.
- Do not replace professional medical advice.
- If the source text mentions or clearly implies that something should be discussed with a healthcare professional, keep this point clear.

Output requirements:
- Write only in the specified output language.
- Use patient-facing language.
- Use a clear structure.
- Use short paragraphs where helpful.
- Use bullet points only where they improve orientation.
- Do not mention the framework, taxonomy, checklist, prompt, or patient profile.
- Return only the simplified text.

Source text:
{source_text}
""".strip()

In [10]:
def load_profile_prompt(profile_name: str) -> str:
    """Load a profile prompt design from the prompts/ folder."""
    path = PROMPT_DIR / f"{profile_name}.txt"
    if not path.exists():
        available = [p.stem for p in sorted(PROMPT_DIR.glob("*.txt"))]
        raise FileNotFoundError(f"Prompt file not found for profile '{profile_name}'. Available: {available}")
    return path.read_text(encoding="utf-8").strip()


def build_generation_messages(
    source_text: str,
    profile_name: str,
    target_language: str = TARGET_LANGUAGE,
) -> List[Dict[str, str]]:
    """Build chat messages for GPT-4o using the shared wrapper and selected profile prompt."""
    if not source_text or not source_text.strip():
        raise ValueError("source_text cannot be empty.")

    profile_prompt = load_profile_prompt(profile_name)
    user_prompt = SHARED_PROMPT_WRAPPER.format(
        target_language=target_language,
        profile_prompt=profile_prompt,
        source_text=source_text.strip()
    )
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_prompt},
    ]


def preview_prompt(source_text: str, profile_name: str, target_language: str = TARGET_LANGUAGE, max_chars: int = 4000) -> None:
    """Print a preview of the generated user prompt."""
    messages = build_generation_messages(source_text, profile_name, target_language)
    print(messages[1]["content"][:max_chars])
    if len(messages[1]["content"]) > max_chars:
        print("\n...[truncated preview]...")

## 6. GPT-4o Generation Function

In [11]:
def generate_simplification(
    source_text: str,
    profile_name: str,
    target_language: str = TARGET_LANGUAGE,
    temperature: float = TEMPERATURE,
    max_tokens: int = MAX_TOKENS,
    run_api_calls: bool = RUN_API_CALLS,
) -> Dict[str, Any]:
    """Generate a patient-profile-specific simplification using Azure OpenAI GPT-4o."""
    messages = build_generation_messages(source_text, profile_name, target_language)

    metadata = {
        "run_id": str(uuid.uuid4()),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "profile_name": profile_name,
        "target_language": target_language,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "deployment_name": AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
        "source_text": source_text,
        "messages": messages,
    }

    if not run_api_calls:
        metadata["simplified_text"] = "[DRY RUN] Set RUN_API_CALLS=True to generate output with Azure OpenAI."
        metadata["api_called"] = False
        return metadata

    azure_client = get_azure_client()
    response = azure_client.chat.completions.create(
        model=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    simplified_text = response.choices[0].message.content

    metadata["simplified_text"] = simplified_text
    metadata["api_called"] = True
    metadata["raw_response_id"] = getattr(response, "id", None)
    return metadata

## 7. Result Saving Utilities

Outputs are saved both as `.jsonl` for reproducibility and optionally as `.csv` for easier review.

In [12]:
def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append a dictionary record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def records_to_csv(jsonl_path: Path, csv_path: Path) -> pd.DataFrame:
    """Convert a JSONL result file to CSV, excluding the full messages field for readability."""
    records = []
    if not jsonl_path.exists():
        raise FileNotFoundError(jsonl_path)
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                record = {k: v for k, v in record.items() if k != "messages"}
                records.append(record)
    df = pd.DataFrame(records)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False, encoding="utf-8")
    return df


def save_generation_record(record: Dict[str, Any], filename: str = "generation_results.jsonl") -> Path:
    path = RESULTS_DIR / filename
    append_jsonl(record, path)
    return path

## 8. Manual Input Workflow

Use this section for one-off simplifications. Change `manual_profile`, `manual_target_language`, and `manual_text`.

In [ ]:
manual_profile = "older_adults"  # choose one of: layperson, non_native_speaker, cognitive_disabilities, older_adults, lower_education_level, general_non_expert_baseline
manual_target_language = "German"

manual_text = """
Über die Nahrung nehmen wir neben anderen Nährstoffen Kohlenhydrate auf, die nach Aufspaltung im Magen-Darm-Trakt als Glukose in das Blut gelangen. Als Folge steigt der Blutglukosespiegel. Die Bauchspeicheldrüse (Pankreas) schüttet eine bedarfsgerechte Menge des Hormons Insulin aus, wodurch die Glukose aus dem Blut in die Körperzellen gelangt.
Bei Menschen mit Diabetes ist die Wirkung von Insulin vermindert oder die Bauchspeicheldrüse produziert zu wenig beziehungsweise kein Insulin mehr, sodass die Blutglukose ansteigt. Auf Basis dieser Störung wird grundsätzlich zwischen Typ-1- und Typ-2-Diabetes unterschieden. Es gibt aber auch seltener auftretende Diabetesformen, die auf anderen Ursachen, beispielweise genetischen Defekten, Infektionen, Stoffwechselstörungen oder chronischen Bauchspeicheldrüsen-Erkrankungen beruhen.
Diabetes Typ 2
Bei Typ-2-Diabetes sprechen die Körperzellen immer schlechter auf das Insulin an. Durch diese sogenannte Insulinresistenz erhöht die Bauchspeicheldrüse zunächst die Insulinproduktion, um das Defizit auszugleichen. Aus diesem Grund ist der Blutglukosewert im nüchternen Zustand bei Personen, die einen Typ-2-Diabetes entwickeln, oft noch im Normbereich. Nach einer gewissen Zeit erschöpft die Bauchspeicheldrüse und kann die Insulinresistenz nicht mehr ausgleichen, weshalb die Blutglukosewerte ansteigen.
Zentrale Risikofaktoren von Typ-2-Diabetes sind Bewegungsmangel, eine energiedichte und ballaststoffarme Ernährung, Adipositas, Rauchen, Gestationsdiabetes, übermäßiger Alkoholkonsum sowie chronischer Stress. Auch genetische Faktoren spielen bei der Entstehung eine Rolle, weshalb Personen mit familiärer Vorbelastung vor allem bei einem gesundheitsschädigenden Verhalten ein erhöhtes Risiko haben, an Typ-2-Diabetes zu erkranken.
Von Prädiabetes spricht man, wenn bereits erhöhte Blutglukosewerte im nüchternen Zustand oder nach dem Essen vorliegen beziehungsweise der Blutglukose-Langzeitwert (HbA1c) erhöht ist.
Menschen mit Prädiabetes, aber auch Personen mit bereits manifestiertem Typ-2-Diabetes, können die erhöhten Blutglukosewerte durch Gewichtsreduktion, mehr Bewegung und eine kalorienreduzierte, gesunde, ausgewogene Ernährung bis in den Normbereich senken. Durch die Umstellung des Lebensstils ist es manchen Betroffenen möglich, eine Behandlung mit Medikamenten oder Insulin zu verzögern oder gar zu vermeiden. Im fortgeschrittenen Stadium von Typ-2-Diabetes ist meist die Einnahme von blutglukosesenkenden Medikamenten in Tablettenform oder zum Spritzen (GLP1-Medikamente/Insulin) notwendig.

""".strip()

preview_prompt(manual_text, manual_profile, manual_target_language, max_chars=30000)

In [ ]:
manual_result = generate_simplification(
    source_text=manual_text,
    profile_name=manual_profile,
    target_language=manual_target_language,
    run_api_calls=RUN_API_CALLS,
)

print(manual_result["simplified_text"])

# Save result
manual_jsonl_path = save_generation_record(manual_result, filename="manual_generation_results.jsonl")
print("Saved to:", manual_jsonl_path)

## 9. Med-EASi Dataset Workflow

This workflow loads examples from the Med-EASi dataset and processes them iteratively.

Expected dataset fields may include expert/source and simple/reference text depending on the dataset version. The helper below tries to detect likely text columns, but you should inspect the dataset once after loading.

In [ ]:
# Uncomment if needed:
# from datasets import load_dataset

In [13]:
def load_medeasi_dataset(split: str = "test"):
    """Load the Med-EASi dataset from Hugging Face.

    Requires the datasets package and internet access.
    """
    try:
        from datasets import load_dataset
    except ImportError as e:
        raise ImportError("Install datasets first: pip install datasets") from e

    # Dataset name used in earlier experiments. Adjust if you use a local copy.
    dataset = load_dataset("cbasu/Med-EASi", split=split)
    return dataset


def inspect_dataset_columns(dataset, n: int = 3) -> pd.DataFrame:
    """Return a small DataFrame preview of the dataset."""
    rows = [dataset[i] for i in range(min(n, len(dataset)))]
    return pd.DataFrame(rows)


def detect_text_columns(columns: List[str]) -> Dict[str, Optional[str]]:
    """Heuristically detect source/reference text columns."""
    lower_map = {c.lower(): c for c in columns}

    source_candidates = [
        "expert", "expert_text", "source", "source_text", "complex", "complex_text",
        "original", "original_text", "input", "text"
    ]
    reference_candidates = [
        "simple", "simple_text", "target", "target_text", "simplified", "simplified_text",
        "reference", "reference_text"
    ]

    source_col = next((lower_map[c] for c in source_candidates if c in lower_map), None)
    reference_col = next((lower_map[c] for c in reference_candidates if c in lower_map), None)

    return {"source_col": source_col, "reference_col": reference_col}

In [ ]:
# Example usage:
medeasi = load_medeasi_dataset(split="test")
preview_df = inspect_dataset_columns(medeasi, n=5)
display(preview_df)
detected = detect_text_columns(list(preview_df.columns))
detected

In [14]:
def run_medeasi_batch(
    dataset,
    profile_name: str,
    target_language: str = TARGET_LANGUAGE,
    source_col: Optional[str] = None,
    reference_col: Optional[str] = None,
    max_examples: Optional[int] = 10,
    start_index: int = 0,
    sleep_seconds: float = 0.2,
    output_filename: Optional[str] = None,
    run_api_calls: bool = RUN_API_CALLS,
) -> Path:
    """Iterate over Med-EASi examples and generate simplifications."""
    if len(dataset) == 0:
        raise ValueError("Dataset is empty.")

    sample = dataset[0]
    columns = list(sample.keys())

    if source_col is None:
        source_col = detect_text_columns(columns)["source_col"]
    if source_col is None:
        raise ValueError(f"Could not detect source text column. Available columns: {columns}")

    if reference_col is None:
        reference_col = detect_text_columns(columns)["reference_col"]

    if output_filename is None:
        output_filename = f"medeasi_{profile_name}_{target_language.lower()}_results.jsonl".replace(" ", "_")

    output_path = RESULTS_DIR / output_filename

    end_index = len(dataset) if max_examples is None else min(len(dataset), start_index + max_examples)

    for idx in tqdm(range(start_index, end_index), desc=f"Processing Med-EASi ({profile_name})"):
        row = dataset[idx]
        source_text = str(row[source_col]).strip()
        reference_text = str(row[reference_col]).strip() if reference_col and row.get(reference_col) is not None else None

        if not source_text:
            continue

        result = generate_simplification(
            source_text=source_text,
            profile_name=profile_name,
            target_language=target_language,
            run_api_calls=run_api_calls,
        )

        result.update({
            "dataset": "Med-EASi",
            "dataset_index": idx,
            "source_col": source_col,
            "reference_col": reference_col,
            "reference_text": reference_text,
            "row_metadata": {k: v for k, v in row.items() if k not in [source_col, reference_col]},
        })

        append_jsonl(result, output_path)
        time.sleep(sleep_seconds)

    return output_path

In [15]:
from pathlib import Path
import json
import html
import pandas as pd
from IPython.display import display, HTML


def load_generation_results(path):
    """
    Load saved generation results from a JSONL, JSON, or CSV file.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if path.suffix.lower() == ".jsonl":
        records = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return pd.DataFrame(records)

    if path.suffix.lower() == ".json":
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        return pd.DataFrame([data])

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file type: {path.suffix}")


def _first_existing(row, possible_names, default=""):
    """
    Return the first non-empty value from a row for several possible column names.
    """
    for name in possible_names:
        if name in row and pd.notna(row[name]) and str(row[name]).strip():
            return str(row[name])
    return default


def display_result_cards(df, start=0, n=3, show_reference=True):
    """
    Display generated simplification results as readable cards in Colab/Jupyter.

    Parameters:
    - df: pandas DataFrame with generation results
    - start: first row index to display
    - n: number of cards to display
    - show_reference: whether to show Med-EASi reference simplification if available
    """
    if df.empty:
        display(HTML("<p><b>No results to display.</b></p>"))
        return

    subset = df.iloc[start:start + n]

    cards_html = """
    <style>
        .tm-card {
            border: 1px solid #ddd;
            border-radius: 12px;
            padding: 18px;
            margin: 18px 0;
            background: #fafafa;
            font-family: Arial, sans-serif;
            line-height: 1.45;
        }
        .tm-meta {
            color: #666;
            font-size: 13px;
            margin-bottom: 12px;
        }
        .tm-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 14px;
        }
        .tm-grid-3 {
            display: grid;
            grid-template-columns: 1fr 1fr 1fr;
            gap: 14px;
        }
        .tm-box {
            background: white;
            border: 1px solid #e3e3e3;
            border-radius: 10px;
            padding: 12px;
            white-space: pre-wrap;
        }
        .tm-title {
            font-weight: bold;
            margin-bottom: 8px;
            color: #222;
        }
        .tm-text {
            font-size: 14px;
            color: #222;
        }
    </style>
    """

    for display_idx, (_, row) in enumerate(subset.iterrows(), start=start):
        source_text = _first_existing(
            row,
            ["source_text", "original_text", "expert_text", "Expert", "input_text", "text"],
            default="[No source text found]"
        )

        simplified_text = _first_existing(
            row,
            ["simplified_text", "generated_text", "output_text", "simplification", "model_output"],
            default="[No generated simplification found]"
        )

        reference_text = _first_existing(
            row,
            ["reference_text", "simple_text", "Simple", "reference", "gold_text"],
            default=""
        )

        profile = _first_existing(row, ["profile_name", "profile", "patient_profile"], default="N/A")
        language = _first_existing(row, ["target_language", "language"], default="N/A")
        dataset_index = _first_existing(row, ["dataset_index", "index", "text_id", "id"], default=str(display_idx))

        source_html = html.escape(source_text)
        simplified_html = html.escape(simplified_text)
        reference_html = html.escape(reference_text)

        if show_reference and reference_text:
            grid_class = "tm-grid-3"
            reference_box = f"""
            <div class="tm-box">
                <div class="tm-title">Med-EASi Reference</div>
                <div class="tm-text">{reference_html}</div>
            </div>
            """
        else:
            grid_class = "tm-grid"
            reference_box = ""

        cards_html += f"""
        <div class="tm-card">
            <div class="tm-meta">
                <b>Result {display_idx}</b> |
                Profile: {html.escape(profile)} |
                Language: {html.escape(language)} |
                ID/Index: {html.escape(dataset_index)}
            </div>

            <div class="{grid_class}">
                <div class="tm-box">
                    <div class="tm-title">Source Text</div>
                    <div class="tm-text">{source_html}</div>
                </div>

                <div class="tm-box">
                    <div class="tm-title">Generated Simplification</div>
                    <div class="tm-text">{simplified_html}</div>
                </div>

                {reference_box}
            </div>
        </div>
        """

    display(HTML(cards_html))


def save_result_review_html(df, output_path, show_reference=True):
    """
    Save all generation results as a standalone HTML file for review.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    html_parts = [
        """
        <!DOCTYPE html>
        <html>
        <head>
            <meta charset="utf-8">
            <title>Talking Medicine Result Review</title>
            <style>
                body {
                    font-family: Arial, sans-serif;
                    margin: 32px;
                    background: #f5f5f5;
                    line-height: 1.45;
                }
                h1 {
                    margin-bottom: 6px;
                }
                .subtitle {
                    color: #666;
                    margin-bottom: 24px;
                }
                .tm-card {
                    border: 1px solid #ddd;
                    border-radius: 12px;
                    padding: 18px;
                    margin: 18px 0;
                    background: #fafafa;
                }
                .tm-meta {
                    color: #666;
                    font-size: 13px;
                    margin-bottom: 12px;
                }
                .tm-grid {
                    display: grid;
                    grid-template-columns: 1fr 1fr;
                    gap: 14px;
                }
                .tm-grid-3 {
                    display: grid;
                    grid-template-columns: 1fr 1fr 1fr;
                    gap: 14px;
                }
                .tm-box {
                    background: white;
                    border: 1px solid #e3e3e3;
                    border-radius: 10px;
                    padding: 12px;
                    white-space: pre-wrap;
                }
                .tm-title {
                    font-weight: bold;
                    margin-bottom: 8px;
                    color: #222;
                }
                .tm-text {
                    font-size: 14px;
                    color: #222;
                }
            </style>
        </head>
        <body>
            <h1>Talking Medicine Result Review</h1>
            <div class="subtitle">Source text, generated simplification, and reference text where available.</div>
        """
    ]

    for idx, row in df.iterrows():
        source_text = _first_existing(
            row,
            ["source_text", "original_text", "expert_text", "Expert", "input_text", "text"],
            default="[No source text found]"
        )

        simplified_text = _first_existing(
            row,
            ["simplified_text", "generated_text", "output_text", "simplification", "model_output"],
            default="[No generated simplification found]"
        )

        reference_text = _first_existing(
            row,
            ["reference_text", "simple_text", "Simple", "reference", "gold_text"],
            default=""
        )

        profile = _first_existing(row, ["profile_name", "profile", "patient_profile"], default="N/A")
        language = _first_existing(row, ["target_language", "language"], default="N/A")
        dataset_index = _first_existing(row, ["dataset_index", "index", "text_id", "id"], default=str(idx))

        if show_reference and reference_text:
            grid_class = "tm-grid-3"
            reference_box = f"""
            <div class="tm-box">
                <div class="tm-title">Med-EASi Reference</div>
                <div class="tm-text">{html.escape(reference_text)}</div>
            </div>
            """
        else:
            grid_class = "tm-grid"
            reference_box = ""

        html_parts.append(f"""
        <div class="tm-card">
            <div class="tm-meta">
                <b>Result {idx}</b> |
                Profile: {html.escape(profile)} |
                Language: {html.escape(language)} |
                ID/Index: {html.escape(dataset_index)}
            </div>

            <div class="{grid_class}">
                <div class="tm-box">
                    <div class="tm-title">Source Text</div>
                    <div class="tm-text">{html.escape(source_text)}</div>
                </div>

                <div class="tm-box">
                    <div class="tm-title">Generated Simplification</div>
                    <div class="tm-text">{html.escape(simplified_text)}</div>
                </div>

                {reference_box}
            </div>
        </div>
        """)

    html_parts.append("""
        </body>
        </html>
    """)

    output_path.write_text("\n".join(html_parts), encoding="utf-8")
    return output_path

In [ ]:
# Example Med-EASi batch run:

medeasi = load_medeasi_dataset(split="test")
output_path = run_medeasi_batch(
    dataset=medeasi,
    profile_name="older_adults",
    target_language="English",
    max_examples=5,
    run_api_calls=RUN_API_CALLS,
)
df_results = records_to_csv(output_path, output_path.with_suffix(".csv"))
#display(df_results.head())
df_preview = load_generation_results(output_path)
print("Loaded results:", len(df_preview))
display(df_preview.head())

display_result_cards(
    df_preview,
    #Adjust which and how many results should be shown
    start=0,
    n=len(df_preview),
    show_reference=True
)

## 10. Evaluation Module

This section evaluates generated simplifications after they have been saved as JSONL files. It contains three complementary components:

1. **BERTScore semantic similarity analysis** between source text and generated simplification.
2. **Checklist-based LLM evaluation** using one evaluation question for each checklist instruction.
3. **Evaluation overview utilities** that combine BERTScore and LLM evaluation results into readable tables and HTML review files.

BERTScore is used as a semantic similarity signal, not as a standalone proof of medical factual correctness. Factual consistency should still be interpreted together with qualitative or expert evaluation.

In [16]:
# ============================================================
# Evaluation Question Bank
# ============================================================

CHECKLIST_EVALUATION_QUESTIONS = {
    # Lexical complexity
    "L2": {
        "group": "Lexical",
        "instruction": "Explain necessary medical terms in familiar language.",
        "question": "Does the generated simplification explain necessary medical terms in familiar language?",
    },
    "L3": {
        "group": "Lexical",
        "instruction": "Clarify abbreviations and acronyms when they may be misunderstood.",
        "question": "Does the generated simplification clarify abbreviations or acronyms that may be misunderstood?",
    },
    "L4": {
        "group": "Lexical",
        "instruction": "Replace unnecessary jargon with simpler expressions where possible.",
        "question": "Does the generated simplification replace unnecessary medical jargon with simpler expressions where this can be done without changing the meaning?",
    },

    # Syntactic complexity
    "S1": {
        "group": "Syntactic",
        "instruction": "Split long or nested sentences into shorter sentences.",
        "question": "Does the generated simplification split long or nested source sentences into shorter, easier-to-process sentences?",
    },
    "S2": {
        "group": "Syntactic",
        "instruction": "Prefer active voice where it makes the meaning clearer.",
        "question": "Does the generated simplification use active voice where it makes the meaning clearer?",
    },
    "S3": {
        "group": "Syntactic",
        "instruction": "Separate different ideas, conditions, or exceptions into distinct sentences.",
        "question": "Does the generated simplification separate different ideas, conditions, or exceptions into distinct sentences or clearly separated units?",
    },

    # Conceptual complexity
    "C2": {
        "group": "Conceptual",
        "instruction": "Add short background explanations for important medical concepts.",
        "question": "Does the generated simplification add short background explanations for important medical concepts where needed?",
    },
    "C3": {
        "group": "Conceptual",
        "instruction": "Make important medical relationships and mechanisms explicit.",
        "question": "Does the generated simplification make important medical relationships or mechanisms explicit, such as connections between symptoms, risks, diagnoses, or treatments?",
    },
    "C4": {
        "group": "Conceptual",
        "instruction": "Use examples or comparisons where they support understanding.",
        "question": "Does the generated simplification use examples or comparisons where they support understanding without introducing unsupported medical information?",
    },

    # Semantic ambiguity
    "A2": {
        "group": "Semantic ambiguity",
        "instruction": "Clarify terms whose medical meaning differs from their everyday meaning.",
        "question": "Does the generated simplification clarify terms whose medical meaning may differ from their everyday meaning?",
    },
    "A3": {
        "group": "Semantic ambiguity",
        "instruction": "Make implied information explicit where misunderstanding is likely.",
        "question": "Does the generated simplification make implied information explicit where misunderstanding would otherwise be likely?",
    },
    "A4": {
        "group": "Semantic ambiguity",
        "instruction": "Clarify vague or relative statements where possible.",
        "question": "Does the generated simplification clarify vague or relative statements, such as unclear statements about risk, severity, or comparison?",
    },

    # Cognitive load
    "CL2": {
        "group": "Cognitive load",
        "instruction": "Group related information together.",
        "question": "Does the generated simplification group related information together in a coherent way?",
    },
    "CL3": {
        "group": "Cognitive load",
        "instruction": "Separate different topics into clear sections or paragraphs. Use headings where appropriate.",
        "question": "Does the generated simplification separate different topics into clear sections or paragraphs, using headings where appropriate?",
    },
    "CL4": {
        "group": "Cognitive load",
        "instruction": "Use step-by-step structure or lists where they improve orientation.",
        "question": "Does the generated simplification use step-by-step structure or lists where they improve orientation for the reader?",
    },

    # Patient-centered complexity
    "P1": {
        "group": "Patient-centered",
        "instruction": "Clarify what the information means for the patient’s own situation.",
        "question": "Does the generated simplification clarify what the information means for the patient’s own situation?",
    },
    "P2": {
        "group": "Patient-centered",
        "instruction": "Make possible options and consequences explicit when they are present in the source text.",
        "question": "Does the generated simplification make possible options and consequences explicit when they are present in the source text?",
    },
    "P3": {
        "group": "Patient-centered",
        "instruction": "State warnings and next steps clearly.",
        "question": "Does the generated simplification state warnings and next steps clearly when they are present or clearly implied in the source text?",
    },
    "P4": {
        "group": "Patient-centered",
        "instruction": "Indicate which points should be discussed with a healthcare professional.",
        "question": "Does the generated simplification indicate which points should be discussed with a healthcare professional when this is present or clearly implied in the source text?",
    },
}

MEDICAL_CONSISTENCY_QUESTIONS = {
    "M1": "Does the generated simplification preserve the main medical meaning of the source text?",
    "M2": "Does the generated simplification avoid adding unsupported diagnoses, symptoms, risks, treatments, warnings, or recommendations?",
    "M3": "Does the generated simplification avoid omitting medically relevant information from the source text?",
    "M4": "Does the generated simplification preserve uncertainty, conditions, warnings, and risk statements?",
    "M5": "Does the generated simplification avoid making uncertain information sound certain?",
    "M6": "Does the generated simplification avoid presenting itself as a replacement for professional medical advice?",
}

# The mapping can be adjusted if the profile prompt designs are changed.
# Identification-only checklist items (L1, C1, A1, CL1) are intentionally excluded,
# because they are not directly assessable from the generated simplification alone.
PROFILE_EVALUATION_ITEMS = {
    "layperson": ["L2", "L3", "L4", "C2", "C3", "C4", "P1", "P2", "P3", "P4"],
    "non_native_speaker": ["L2", "L3", "S1", "S2", "S3", "A2", "A3", "A4"],
    "cognitive_disabilities": ["CL2", "CL3", "CL4", "S1", "S3", "L2", "L3", "A3"],
    "older_adults": ["S1", "S2", "S3", "CL2", "CL3", "CL4", "P1", "P2", "P3", "P4", "L2", "L3", "A2", "A3", "A4"],
    "lower_education_level": ["L2", "L3", "L4", "C2", "C3", "C4", "S1", "S2", "S3", "P1", "P2", "P3", "P4"],
    "general_non_expert_baseline": list(CHECKLIST_EVALUATION_QUESTIONS.keys()),
}

SCORE_SCALE_DESCRIPTION = "0 = not fulfilled, 1 = partly fulfilled, 2 = fully fulfilled, null = not applicable to the source text"

print("Checklist evaluation questions:", len(CHECKLIST_EVALUATION_QUESTIONS))
print("Medical consistency questions:", len(MEDICAL_CONSISTENCY_QUESTIONS))



Checklist evaluation questions: 19
Medical consistency questions: 6


### 10.1 BERTScore Semantic Similarity Analyzer

This cell compares the original source text with the generated simplification. The generated simplification is treated as the candidate, and the source text is treated as the reference. BERTScore precision, recall, and F1 are saved back into the same folder as the selected generation file.

In [17]:
# ============================================================
# BERTScore Evaluation
# ============================================================

from pathlib import Path
import json
import pandas as pd


def load_jsonl_as_dataframe(path: Path) -> pd.DataFrame:
    """Load a JSONL file into a pandas DataFrame."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


def save_dataframe_as_jsonl(df: pd.DataFrame, path: Path) -> Path:
    """Save a pandas DataFrame as JSONL."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in df.to_dict(orient="records"):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return path


def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str:
    """Find the first column name that exists in the DataFrame."""
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"None of the expected columns were found. Tried: {candidates}. "
        f"Available columns: {df.columns.tolist()}"
    )


def normalize_bertscore_language(language: str | None) -> str:
    """Convert common language names into BERTScore language codes."""
    if language is None:
        return "en"

    lang = str(language).strip().lower()
    mapping = {
        "english": "en",
        "en": "en",
        "german": "de",
        "deutsch": "de",
        "de": "de",
        "french": "fr",
        "français": "fr",
        "fr": "fr",
        "italian": "it",
        "italiano": "it",
        "it": "it",
        "spanish": "es",
        "español": "es",
        "es": "es",
    }
    return mapping.get(lang, lang)


def run_bertscore_evaluation(
    generations_path: Path,
    output_language: str = TARGET_LANGUAGE,
    output_prefix: str = "bertscore_evaluated",
    batch_size: int = 16,
    rescale_with_baseline: bool = True,
) -> tuple[pd.DataFrame, Path, Path]:
    """
    Compute BERTScore between source texts and generated simplifications.

    The generated simplification is treated as the candidate.
    The original source text is treated as the reference.

    Important: BERTScore is most meaningful when source and generated text are in the same language.
    """
    try:
        from bert_score import score as bert_score
    except ImportError as e:
        raise ImportError("Install bert-score first: pip install bert-score") from e

    generations_path = Path(generations_path)
    run_dir = generations_path.parent

    df = load_jsonl_as_dataframe(generations_path)
    if df.empty:
        raise ValueError("The generations file is empty.")

    source_col = first_existing_column(df, ["source_text", "original_text", "expert_text", "Expert", "input_text", "text"])
    simplified_col = first_existing_column(df, ["simplified_text", "generated_text", "output_text", "simplification", "model_output"])

    sources = df[source_col].fillna("").astype(str).tolist()
    simplifications = df[simplified_col].fillna("").astype(str).tolist()
    bert_lang = normalize_bertscore_language(output_language)

    print(f"Loaded {len(df)} generated simplifications.")
    print(f"Source column: {source_col}")
    print(f"Simplification column: {simplified_col}")
    print(f"BERTScore language: {bert_lang}")

    precision, recall, f1 = bert_score(
        cands=simplifications,
        refs=sources,
        lang=bert_lang,
        batch_size=batch_size,
        verbose=True,
        rescale_with_baseline=rescale_with_baseline,
    )

    df["bertscore_precision"] = [float(x) for x in precision.tolist()]
    df["bertscore_recall"] = [float(x) for x in recall.tolist()]
    df["bertscore_f1"] = [float(x) for x in f1.tolist()]

    output_jsonl = run_dir / f"{output_prefix}.jsonl"
    output_csv = run_dir / f"{output_prefix}.csv"

    save_dataframe_as_jsonl(df, output_jsonl)
    df.to_csv(output_csv, index=False, encoding="utf-8")

    print("BERTScore evaluation complete.")
    print("Saved JSONL:", output_jsonl)
    print("Saved CSV:", output_csv)

    display(df[["bertscore_precision", "bertscore_recall", "bertscore_f1"]].describe())

    return df, output_jsonl, output_csv

In [ ]:
# Example BERTScore run.
# Adjust generations_path to the JSONL file you want to evaluate.

# generations_path = RESULTS_DIR / "medeasi_older_adults_english_results.jsonl"

# df_bertscore, bertscore_jsonl_path, bertscore_csv_path = run_bertscore_evaluation(
#     generations_path=generations_path,
#     output_language="English",
#     output_prefix="bertscore_evaluated",
#     batch_size=16,
# )

In [48]:
# ============================================================
# Run BERTScore Evaluation on a Selected Generation Run
# ============================================================

from pathlib import Path

# Option 1: automatically use the latest run folder
# run_dir = sorted(Path("/content/outputs/runs").glob("*"), reverse=True)[0]

# Option 2: manually select a specific run folder
run_dir = Path("./outputs/runs/cognitive_disabilities")

generations_path = run_dir / "generations.jsonl"

print("Selected run folder:", run_dir)
print("Generations file:", generations_path)

df_bertscore, bertscore_jsonl_path, bertscore_csv_path = run_bertscore_evaluation(
    generations_path=generations_path,
    output_language="English",   # change only if source + simplification are both another language
    output_prefix="bertscore_evaluated",
    batch_size=16,
)

print("BERTScore JSONL saved to:", bertscore_jsonl_path)
print("BERTScore CSV saved to:", bertscore_csv_path)

display(df_bertscore[[
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1"
]].head())

Selected run folder: outputs\runs\cognitive_disabilities
Generations file: outputs\runs\cognitive_disabilities\generations.jsonl
Loaded 300 generated simplifications.
Source column: source_text
Simplification column: simplified_text
BERTScore language: en


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11578.83it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|██████████| 38/38 [00:38<00:00,  1.01s/it]


computing greedy matching.


100%|██████████| 19/19 [00:00<00:00, 274.37it/s]


done in 38.43 seconds, 7.81 sentences/sec
BERTScore evaluation complete.
Saved JSONL: outputs\runs\cognitive_disabilities\bertscore_evaluated.jsonl
Saved CSV: outputs\runs\cognitive_disabilities\bertscore_evaluated.csv


,bertscore_precision,bertscore_recall,bertscore_f1
count,300.000000,300.000000,300.000000
mean,0.208012,0.447984,0.324727
std,0.236369,0.180372,0.191114
min,-0.538663,-0.136146,-0.205549
25%,0.056784,0.334513,0.195904
50%,0.176209,0.457903,0.308405
75%,0.348788,0.564707,0.447571
max,0.802116,0.866685,0.828565


BERTScore JSONL saved to: outputs\runs\cognitive_disabilities\bertscore_evaluated.jsonl
BERTScore CSV saved to: outputs\runs\cognitive_disabilities\bertscore_evaluated.csv


,bertscore_precision,bertscore_recall,bertscore_f1
0,0.249261,0.561668,0.401876
1,0.113168,0.267215,0.190355
2,-0.194873,0.237926,0.013694
3,-0.538663,0.177552,-0.205549
4,-0.013607,0.446977,0.207690


### 10.2 LLM-Based Checklist Evaluation

This evaluator asks GPT-4o to score the generated simplification using the checklist questions and separate medical consistency questions. The selected patient profile determines which checklist questions are used. Medical consistency questions are evaluated for every output.

In [49]:
# ============================================================
# LLM-Based Checklist Evaluation
# ============================================================

import re
from typing import Any

EVALUATION_SYSTEM_MESSAGE = """
You are a strict evaluator of medical text simplification.
Assess the generated simplification against the source text and the evaluation questions.
Return valid JSON only. Do not include markdown, explanations outside JSON, or code fences.
""".strip()

EVALUATION_PROMPT_TEMPLATE = """
Evaluate the generated medical simplification.

Scoring scale:
{score_scale}

Patient profile:
{profile_name}

Checklist evaluation questions:
{checklist_questions}

Medical consistency questions:
{medical_questions}

Source text:
{source_text}

Generated simplification:
{simplified_text}

Return valid JSON in exactly this structure:
{{
  "checklist_scores": {{
    "ITEM_CODE": {{
      "score": 0 or 1 or 2 or null,
      "applicable": true or false,
      "reason": "short reason"
    }}
  }},
  "medical_consistency_scores": {{
    "ITEM_CODE": {{
      "score": 0 or 1 or 2 or null,
      "applicable": true or false,
      "reason": "short reason"
    }}
  }},
  "overall_checklist_comment": "short comment",
  "overall_medical_consistency_comment": "short comment"
}}
""".strip()


def format_evaluation_questions(profile_name: str) -> str:
    """Format profile-specific checklist questions for the evaluator."""
    item_codes = PROFILE_EVALUATION_ITEMS.get(profile_name, list(CHECKLIST_EVALUATION_QUESTIONS.keys()))
    lines = []
    for code in item_codes:
        item = CHECKLIST_EVALUATION_QUESTIONS[code]
        lines.append(f"{code} ({item['group']}): {item['question']}")
    return "\n".join(lines)


def format_medical_questions() -> str:
    """Format medical consistency questions for the evaluator."""
    return "\n".join([f"{code}: {question}" for code, question in MEDICAL_CONSISTENCY_QUESTIONS.items()])


def extract_json_object(text: str) -> dict[str, Any]:
    """Extract and parse a JSON object from a model response."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise


def build_llm_evaluation_messages(
    source_text: str,
    simplified_text: str,
    profile_name: str,
) -> list[dict[str, str]]:
    user_prompt = EVALUATION_PROMPT_TEMPLATE.format(
        score_scale=SCORE_SCALE_DESCRIPTION,
        profile_name=profile_name,
        checklist_questions=format_evaluation_questions(profile_name),
        medical_questions=format_medical_questions(),
        source_text=source_text.strip(),
        simplified_text=simplified_text.strip(),
    )
    return [
        {"role": "system", "content": EVALUATION_SYSTEM_MESSAGE},
        {"role": "user", "content": user_prompt},
    ]


def evaluate_simplification_with_llm(
    source_text: str,
    simplified_text: str,
    profile_name: str,
    generation_record: dict[str, Any] | None = None,
    temperature: float = 0.0,
    max_tokens: int = 1800,
    run_api_calls: bool = RUN_API_CALLS,
) -> dict[str, Any]:
    """Evaluate one generated simplification with GPT-4o as checklist evaluator."""
    messages = build_llm_evaluation_messages(source_text, simplified_text, profile_name)

    eval_record = {
        "evaluation_id": str(uuid.uuid4()),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "profile_name": profile_name,
        "source_run_id": generation_record.get("run_id") if generation_record else None,
        "source_dataset": generation_record.get("dataset") if generation_record else None,
        "source_dataset_index": generation_record.get("dataset_index") if generation_record else None,
        "evaluation_item_codes": PROFILE_EVALUATION_ITEMS.get(profile_name, list(CHECKLIST_EVALUATION_QUESTIONS.keys())),
        "medical_item_codes": list(MEDICAL_CONSISTENCY_QUESTIONS.keys()),
        "messages": messages,
    }

    if not run_api_calls:
        eval_record["api_called"] = False
        eval_record["evaluation_raw"] = "[DRY RUN] Set RUN_API_CALLS=True to run LLM evaluation."
        eval_record["evaluation_json"] = None
        return eval_record

    azure_client = get_azure_client()
    response = azure_client.chat.completions.create(
        model=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    raw = response.choices[0].message.content
    eval_record["api_called"] = True
    eval_record["evaluation_raw"] = raw
    eval_record["raw_response_id"] = getattr(response, "id", None)

    try:
        eval_record["evaluation_json"] = extract_json_object(raw)
        eval_record["parse_error"] = None
    except Exception as e:
        eval_record["evaluation_json"] = None
        eval_record["parse_error"] = str(e)

    return eval_record


def evaluate_generation_jsonl_with_llm(
    generation_jsonl_path: Path,
    output_filename: str = "llm_checklist_evaluations.jsonl",
    max_records: int | None = None,
    run_api_calls: bool = RUN_API_CALLS,
    sleep_seconds: float = 0.2,
) -> Path:
    """Evaluate saved generation results with the LLM checklist evaluator."""
    generation_jsonl_path = Path(generation_jsonl_path)
    output_path = generation_jsonl_path.parent / output_filename

    records = []
    with generation_jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    if max_records is not None:
        records = records[:max_records]

    for record in tqdm(records, desc="LLM checklist evaluation"):
        source_text = str(record.get("source_text", "")).strip()
        simplified_text = str(record.get("simplified_text", "")).strip()
        profile_name = str(record.get("profile_name", "general_non_expert_baseline")).strip()

        if not source_text or not simplified_text:
            continue

        eval_record = evaluate_simplification_with_llm(
            source_text=source_text,
            simplified_text=simplified_text,
            profile_name=profile_name,
            generation_record=record,
            run_api_calls=run_api_calls,
        )
        append_jsonl(eval_record, output_path)
        time.sleep(sleep_seconds)

    print("Saved LLM checklist evaluations to:", output_path)
    return output_path

In [ ]:
# Example LLM checklist evaluation run.
# Adjust generation_jsonl_path to the JSONL file you want to evaluate.

# generation_jsonl_path = RESULTS_DIR / "medeasi_older_adults_english_results.jsonl"
#
# llm_eval_path = evaluate_generation_jsonl_with_llm(
#     generation_jsonl_path=generation_jsonl_path,
#     output_filename="llm_checklist_evaluations.jsonl",
#     max_records=5,          # set to None for the whole file
#     run_api_calls=RUN_API_CALLS,
# )

In [10]:
# ============================================================
# Run LLM Checklist Evaluation on a Selected Generation Run
# ============================================================

from pathlib import Path

# Option 1: automatically use the latest run folder
# run_dir = sorted(Path("/content/outputs/runs").glob("*"), reverse=True)[0]

# Option 2: manually select a specific run folder
run_dir = Path("./outputs/runs/cognitive_disabilities")

generations_path = run_dir / "generations.jsonl"

print("Selected run folder:", run_dir)
print("Generations file:", generations_path)

# For testing, start with a small number.
# Set max_records=None later to evaluate the whole generation file.
llm_eval_path = evaluate_generation_jsonl_with_llm(
    generation_jsonl_path=generations_path,
    output_filename="llm_checklist_evaluations.jsonl",
    max_records=301,          # change to None for the whole run
    run_api_calls=RUN_API_CALLS,
    sleep_seconds=0.2,
)

print("LLM checklist evaluation saved to:", llm_eval_path)

Selected run folder: outputs\runs\cognitive_disabilities
Generations file: outputs\runs\cognitive_disabilities\generations.jsonl


NameError: name 'evaluate_generation_jsonl_with_llm' is not defined

### 10.3 Evaluation Overview

The following utilities create a readable overview from BERTScore and LLM checklist evaluation outputs. They summarize per-item scores, aggregate checklist scores, aggregate medical consistency scores, and semantic similarity scores.

In [ ]:
# ============================================================
# Evaluation Overview Utilities
# ============================================================

from IPython.display import display, HTML
import html


def score_dict_mean(score_dict: dict[str, Any] | None) -> float | None:
    """Compute mean score over applicable items in a score dictionary."""
    if not score_dict:
        return None
    scores = []
    for item in score_dict.values():
        if not isinstance(item, dict):
            continue
        if item.get("applicable") is False:
            continue
        score = item.get("score")
        if score is not None:
            scores.append(float(score))
    if not scores:
        return None
    return sum(scores) / len(scores)


def count_applicable_items(score_dict: dict[str, Any] | None) -> int:
    """Count applicable items in a score dictionary."""
    if not score_dict:
        return 0
    count = 0
    for item in score_dict.values():
        if isinstance(item, dict) and item.get("applicable") is not False and item.get("score") is not None:
            count += 1
    return count


def flatten_llm_evaluations(llm_eval_path: Path) -> pd.DataFrame:
    """Load and flatten LLM checklist evaluations."""
    df = load_jsonl_as_dataframe(llm_eval_path)
    rows = []
    for _, row in df.iterrows():
        evaluation_json = row.get("evaluation_json")
        if isinstance(evaluation_json, str):
            try:
                evaluation_json = json.loads(evaluation_json)
            except Exception:
                evaluation_json = None

        checklist_scores = (evaluation_json or {}).get("checklist_scores") if evaluation_json else None
        medical_scores = (evaluation_json or {}).get("medical_consistency_scores") if evaluation_json else None

        rows.append({
            "source_run_id": row.get("source_run_id"),
            "profile_name": row.get("profile_name"),
            "source_dataset": row.get("source_dataset"),
            "source_dataset_index": row.get("source_dataset_index"),
            "checklist_mean_score": score_dict_mean(checklist_scores),
            "checklist_applicable_items": count_applicable_items(checklist_scores),
            "medical_consistency_mean_score": score_dict_mean(medical_scores),
            "medical_consistency_applicable_items": count_applicable_items(medical_scores),
            "overall_checklist_comment": (evaluation_json or {}).get("overall_checklist_comment") if evaluation_json else None,
            "overall_medical_consistency_comment": (evaluation_json or {}).get("overall_medical_consistency_comment") if evaluation_json else None,
            "parse_error": row.get("parse_error"),
        })
    return pd.DataFrame(rows)


def build_evaluation_overview(
    generations_path: Path,
    bertscore_path: Path | None = None,
    llm_eval_path: Path | None = None,
) -> pd.DataFrame:
    """Combine generation records, optional BERTScore results, and optional LLM evaluation summaries."""
    generations_path = Path(generations_path)
    gen_df = load_jsonl_as_dataframe(generations_path)

    # Keep readable columns from the original generations.
    keep_cols = [
        col for col in [
            "run_id", "timestamp_utc", "profile_name", "target_language", "dataset", "dataset_index",
            "source_text", "simplified_text", "reference_text", "api_called", "deployment_name"
        ] if col in gen_df.columns
    ]
    overview = gen_df[keep_cols].copy()

    if bertscore_path is not None:
        bert_df = load_jsonl_as_dataframe(bertscore_path)
        bert_cols = ["run_id", "bertscore_precision", "bertscore_recall", "bertscore_f1"]
        bert_cols = [col for col in bert_cols if col in bert_df.columns]
        if "run_id" in bert_cols and "run_id" in overview.columns:
            overview = overview.merge(bert_df[bert_cols], on="run_id", how="left")

    if llm_eval_path is not None:
        llm_df = flatten_llm_evaluations(llm_eval_path)
        if "run_id" in overview.columns and "source_run_id" in llm_df.columns:
            overview = overview.merge(llm_df, left_on="run_id", right_on="source_run_id", how="left", suffixes=("", "_eval"))

    return overview


def display_evaluation_summary(overview_df: pd.DataFrame) -> None:
    """Display aggregate evaluation statistics."""
    numeric_cols = [
        col for col in [
            "bertscore_precision", "bertscore_recall", "bertscore_f1",
            "checklist_mean_score", "medical_consistency_mean_score"
        ] if col in overview_df.columns
    ]

    print("Number of evaluated records:", len(overview_df))
    if numeric_cols:
        display(overview_df[numeric_cols].describe())

    group_cols = [col for col in ["profile_name", "target_language"] if col in overview_df.columns]
    if group_cols and numeric_cols:
        display(
            overview_df.groupby(group_cols)[numeric_cols]
            .mean(numeric_only=True)
            .reset_index()
        )


def display_evaluation_cards(overview_df: pd.DataFrame, start: int = 0, n: int = 5) -> None:
    """Display evaluation records as readable cards."""
    subset = overview_df.iloc[start:start + n]

    style = """
    <style>
        .eval-card { border:1px solid #ddd; border-radius:12px; padding:16px; margin:16px 0; background:#fafafa; font-family:Arial, sans-serif; }
        .eval-meta { color:#555; font-size:13px; margin-bottom:10px; }
        .eval-grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; }
        .eval-box { background:#fff; border:1px solid #e3e3e3; border-radius:10px; padding:12px; white-space:pre-wrap; }
        .eval-title { font-weight:bold; margin-bottom:6px; }
        .eval-scores { background:#f1f1f1; border-radius:10px; padding:10px; margin-bottom:12px; }
    </style>
    """
    body = style

    for idx, row in subset.iterrows():
        def get_value(name, default=""):
            value = row.get(name, default)
            if pd.isna(value):
                return default
            return value

        source_text = str(get_value("source_text", "[No source text]"))
        simplified_text = str(get_value("simplified_text", "[No simplification]"))
        profile = str(get_value("profile_name", "N/A"))
        dataset_index = str(get_value("dataset_index", idx))

        score_lines = []
        for label, col in [
            ("BERTScore P", "bertscore_precision"),
            ("BERTScore R", "bertscore_recall"),
            ("BERTScore F1", "bertscore_f1"),
            ("Checklist mean", "checklist_mean_score"),
            ("Medical consistency mean", "medical_consistency_mean_score"),
        ]:
            if col in row and pd.notna(row[col]):
                score_lines.append(f"<b>{label}:</b> {float(row[col]):.3f}")

        checklist_comment = str(get_value("overall_checklist_comment", ""))
        medical_comment = str(get_value("overall_medical_consistency_comment", ""))

        body += f"""
        <div class="eval-card">
            <div class="eval-meta"><b>Record {html.escape(str(idx))}</b> | Profile: {html.escape(profile)} | Dataset index: {html.escape(dataset_index)}</div>
            <div class="eval-scores">{' | '.join(score_lines) if score_lines else 'No evaluation scores available.'}</div>
            <div class="eval-grid">
                <div class="eval-box"><div class="eval-title">Source Text</div>{html.escape(source_text)}</div>
                <div class="eval-box"><div class="eval-title">Generated Simplification</div>{html.escape(simplified_text)}</div>
            </div>
            {f'<div class="eval-box" style="margin-top:12px;"><div class="eval-title">Checklist Comment</div>{html.escape(checklist_comment)}</div>' if checklist_comment else ''}
            {f'<div class="eval-box" style="margin-top:12px;"><div class="eval-title">Medical Consistency Comment</div>{html.escape(medical_comment)}</div>' if medical_comment else ''}
        </div>
        """

    display(HTML(body))


def save_evaluation_overview_html(overview_df: pd.DataFrame, output_path: Path) -> Path:
    """Save a standalone HTML evaluation overview file."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    

    rows_html = []
    for idx, row in overview_df.iterrows():
        def esc(value):
            if pd.isna(value):
                return ""
            return html.escape(str(value))

        scores = []
        for label, col in [
            ("BERT P", "bertscore_precision"), ("BERT R", "bertscore_recall"), ("BERT F1", "bertscore_f1"),
            ("Checklist", "checklist_mean_score"), ("Medical", "medical_consistency_mean_score"),
        ]:
            if col in overview_df.columns and pd.notna(row.get(col)):
                scores.append(f"<b>{label}:</b> {float(row.get(col)):.3f}")

        rows_html.append(f"""
        <div class="card">
            <div class="meta"><b>Record {idx}</b> | Profile: {esc(row.get('profile_name'))} | Dataset index: {esc(row.get('dataset_index'))}</div>
            <div class="scores">{' | '.join(scores) if scores else 'No scores available.'}</div>
            <div class="grid">
                <div class="box"><h3>Source Text</h3>{esc(row.get('source_text'))}</div>
                <div class="box"><h3>Generated Simplification</h3>{esc(row.get('simplified_text'))}</div>
            </div>
            <div class="box"><h3>Checklist Comment</h3>{esc(row.get('overall_checklist_comment'))}</div>
            <div class="box"><h3>Medical Consistency Comment</h3>{esc(row.get('overall_medical_consistency_comment'))}</div>
        </div>
        """)

    html_doc = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Talking Medicine Evaluation Overview</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 32px; background:#f5f5f5; line-height:1.45; }}
            .card {{ border:1px solid #ddd; border-radius:12px; padding:18px; margin:18px 0; background:#fafafa; }}
            .meta {{ color:#555; font-size:13px; margin-bottom:10px; }}
            .scores {{ background:#f1f1f1; border-radius:10px; padding:10px; margin-bottom:12px; }}
            .grid {{ display:grid; grid-template-columns:1fr 1fr; gap:12px; }}
            .box {{ background:#fff; border:1px solid #e3e3e3; border-radius:10px; padding:12px; white-space:pre-wrap; margin-top:12px; }}
            h1 {{ margin-bottom:4px; }}
            h3 {{ margin-top:0; }}
        </style>
    </head>
    <body>
        <h1>Talking Medicine Evaluation Overview</h1>
        <p>Combined overview of semantic similarity, checklist-based evaluation, and medical consistency evaluation.</p>
        {''.join(rows_html)}
    </body>
    </html>
    """
    output_path.write_text(html_doc, encoding="utf-8")
    print("Saved evaluation overview HTML:", output_path)
    return output_path

In [36]:
# ============================================================
# Save Combined Evaluation Report as HTML
# Compatible with evaluation_json -> checklist_scores structure
# ============================================================

from pathlib import Path
import json
import html


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        print(f"File not found: {path}")
        return []

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def get_first(record, keys, default=""):
    for key in keys:
        if key in record and record[key] not in [None, ""]:
            return record[key]
    return default


def get_eval_root(eval_record):
    """
    Your LLM evaluation file stores the parsed scores under 'evaluation_json'.
    """
    if not isinstance(eval_record, dict):
        return {}

    if isinstance(eval_record.get("evaluation_json"), dict):
        return eval_record["evaluation_json"]

    if isinstance(eval_record.get("parsed_response"), dict):
        return eval_record["parsed_response"]

    if isinstance(eval_record.get("response"), dict):
        return eval_record["response"]

    return eval_record


def extract_score_items(eval_record, score_key):
    """
    Extracts items from:
    evaluation_json -> checklist_scores
    evaluation_json -> medical_consistency_scores
    """
    root = get_eval_root(eval_record)
    score_dict = root.get(score_key, {})

    if not isinstance(score_dict, dict):
        return []

    items = []
    for item_id, item_data in score_dict.items():
        if isinstance(item_data, dict):
            items.append({
                "item_id": item_id,
                "score": item_data.get("score"),
                "applicable": item_data.get("applicable"),
                "reason": item_data.get("reason", ""),
            })

    return items


def get_eval_comment(eval_record, key, default=""):
    root = get_eval_root(eval_record)
    return root.get(key, default)


def format_score(score):
    if score is None:
        return "N/A"
    return str(score)


def mean_applicable_score(items):
    values = [
        item["score"]
        for item in items
        if isinstance(item.get("score"), (int, float))
    ]
    if not values:
        return None
    return sum(values) / len(values)


def save_combined_evaluation_html(
    run_dir,
    generations_filename="generations.jsonl",
    bertscore_filename="bertscore_evaluated.jsonl",
    llm_eval_filename="llm_checklist_evaluations.jsonl",
    output_filename="evaluation_overview.html",
):
    run_dir = Path(run_dir)

    generations_path = run_dir / generations_filename
    bertscore_path = run_dir / bertscore_filename
    llm_eval_path = run_dir / llm_eval_filename

    generations = load_jsonl(generations_path)
    bertscore_records = load_jsonl(bertscore_path)
    llm_eval_records = load_jsonl(llm_eval_path)

    print("Loaded generation records:", len(generations))
    print("Loaded BERTScore records:", len(bertscore_records))
    print("Loaded LLM evaluation records:", len(llm_eval_records))

    if not generations:
        raise ValueError(f"No generation records found in {generations_path}")

    # Diagnostic check
    if llm_eval_records:
        first_eval_root = get_eval_root(llm_eval_records[0])
        print("First LLM evaluation root keys:", list(first_eval_root.keys()))
        print("Checklist items in first LLM eval:", len(extract_score_items(llm_eval_records[0], "checklist_scores")))
        print("Medical items in first LLM eval:", len(extract_score_items(llm_eval_records[0], "medical_consistency_scores")))

    bert_f1_values = []
    checklist_means = []
    medical_means = []

    for i, gen in enumerate(generations):
        if i < len(bertscore_records):
            f1 = get_first(bertscore_records[i], ["bertscore_f1"], None)
            if isinstance(f1, (int, float)):
                bert_f1_values.append(f1)

        if i < len(llm_eval_records):
            checklist_items = extract_score_items(llm_eval_records[i], "checklist_scores")
            medical_items = extract_score_items(llm_eval_records[i], "medical_consistency_scores")

            checklist_mean = mean_applicable_score(checklist_items)
            medical_mean = mean_applicable_score(medical_items)

            if checklist_mean is not None:
                checklist_means.append(checklist_mean)
            if medical_mean is not None:
                medical_means.append(medical_mean)

    def mean_or_na(values):
        return round(sum(values) / len(values), 3) if values else "N/A"

    html_parts = ["""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Talking Medicine Evaluation Overview</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 32px;
                background: #f5f5f5;
                color: #222;
                line-height: 1.45;
            }
            h1 { margin-bottom: 4px; }
            .subtitle {
                color: #666;
                margin-bottom: 24px;
            }
            .summary {
                background: white;
                border: 1px solid #ddd;
                border-radius: 12px;
                padding: 16px;
                margin-bottom: 24px;
            }
            .card {
                background: #fafafa;
                border: 1px solid #ddd;
                border-radius: 14px;
                padding: 18px;
                margin: 22px 0;
            }
            .meta {
                color: #666;
                font-size: 13px;
                margin-bottom: 12px;
            }
            .grid-2 {
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 14px;
                margin-bottom: 14px;
            }
            .grid-3 {
                display: grid;
                grid-template-columns: 1fr 1fr 1fr;
                gap: 14px;
                margin-bottom: 14px;
            }
            .box {
                background: white;
                border: 1px solid #e3e3e3;
                border-radius: 10px;
                padding: 12px;
                white-space: pre-wrap;
            }
            .box-title {
                font-weight: bold;
                margin-bottom: 8px;
            }
            .metric-row {
                display: flex;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 10px;
            }
            .metric {
                background: white;
                border: 1px solid #e3e3e3;
                border-radius: 10px;
                padding: 10px 12px;
                min-width: 120px;
            }
            .metric-label {
                font-size: 12px;
                color: #666;
            }
            .metric-value {
                font-size: 18px;
                font-weight: bold;
                margin-top: 4px;
            }
            table {
                width: 100%;
                border-collapse: collapse;
                background: white;
                margin-top: 8px;
                margin-bottom: 14px;
            }
            th, td {
                border: 1px solid #e3e3e3;
                padding: 8px;
                vertical-align: top;
                font-size: 13px;
            }
            th {
                background: #f0f0f0;
                text-align: left;
            }
            .score {
                font-weight: bold;
                text-align: center;
                width: 60px;
            }
            .section-title {
                font-weight: bold;
                margin-top: 16px;
                margin-bottom: 8px;
                font-size: 16px;
            }
            .missing {
                color: #9a6700;
                background: #fff8db;
                border: 1px solid #f0d98c;
                border-radius: 8px;
                padding: 8px;
            }
        </style>
    </head>
    <body>
        <h1>Talking Medicine Evaluation Overview</h1>
        <div class="subtitle">
            Combined view of source texts, generated simplifications, BERTScore values, checklist evaluation, and medical consistency evaluation.
        </div>
    """]

    html_parts.append(f"""
        <div class="summary">
            <div class="section-title">Run Summary</div>
            <div class="metric-row">
                <div class="metric">
                    <div class="metric-label">Generated examples</div>
                    <div class="metric-value">{len(generations)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">BERTScore records</div>
                    <div class="metric-value">{len(bertscore_records)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">LLM evaluation records</div>
                    <div class="metric-value">{len(llm_eval_records)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Mean BERTScore F1</div>
                    <div class="metric-value">{mean_or_na(bert_f1_values)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Mean checklist score</div>
                    <div class="metric-value">{mean_or_na(checklist_means)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Mean medical consistency score</div>
                    <div class="metric-value">{mean_or_na(medical_means)}</div>
                </div>
            </div>
        </div>
    """)

    def items_table(items, empty_message):
        if not items:
            return f'<div class="missing">{html.escape(empty_message)}</div>'

        rows = ""
        for item in items:
            item_id = html.escape(str(item.get("item_id", "")))
            score = html.escape(format_score(item.get("score")))
            applicable = item.get("applicable")
            applicable_text = "Yes" if applicable is True else "No" if applicable is False else "N/A"
            reason = html.escape(str(item.get("reason", "")))

            rows += f"""
            <tr>
                <td>{item_id}</td>
                <td class="score">{score}</td>
                <td>{applicable_text}</td>
                <td>{reason}</td>
            </tr>
            """

        return f"""
        <table>
            <thead>
                <tr>
                    <th>Item</th>
                    <th>Score</th>
                    <th>Applicable</th>
                    <th>Reason</th>
                </tr>
            </thead>
            <tbody>
                {rows}
            </tbody>
        </table>
        """

    for i, gen in enumerate(generations):
        bert = bertscore_records[i] if i < len(bertscore_records) else {}
        llm_eval = llm_eval_records[i] if i < len(llm_eval_records) else {}

        source_text = get_first(gen, ["source_text", "original_text", "expert_text", "Expert", "input_text", "text"], "[No source text found]")
        simplified_text = get_first(gen, ["simplified_text", "generated_text", "output_text", "simplification", "model_output"], "[No simplification found]")
        reference_text = get_first(gen, ["reference_text", "simple_text", "Simple", "reference", "gold_text"], "")

        profile = get_first(gen, ["profile_name", "profile", "patient_profile"], "N/A")
        target_language = get_first(gen, ["target_language", "language"], "N/A")
        dataset_index = get_first(gen, ["dataset_index", "index", "text_id", "id"], str(i))

        precision = get_first(bert, ["bertscore_precision"], "N/A")
        recall = get_first(bert, ["bertscore_recall"], "N/A")
        f1 = get_first(bert, ["bertscore_f1"], "N/A")

        if isinstance(precision, float):
            precision = round(precision, 3)
        if isinstance(recall, float):
            recall = round(recall, 3)
        if isinstance(f1, float):
            f1 = round(f1, 3)

        checklist_items = extract_score_items(llm_eval, "checklist_scores")
        medical_items = extract_score_items(llm_eval, "medical_consistency_scores")

        overall_checklist_comment = get_eval_comment(
            llm_eval,
            "overall_checklist_comment",
            ""
        )

        overall_medical_comment = get_eval_comment(
            llm_eval,
            "overall_medical_consistency_comment",
            ""
        )

        reference_box = ""
        grid_class = "grid-2"

        if reference_text:
            grid_class = "grid-3"
            reference_box = f"""
            <div class="box">
                <div class="box-title">Reference Simplification</div>
                {html.escape(str(reference_text))}
            </div>
            """

        html_parts.append(f"""
        <div class="card">
            <div class="meta">
                <b>Example {i}</b> |
                Profile: {html.escape(str(profile))} |
                Language: {html.escape(str(target_language))} |
                ID/Index: {html.escape(str(dataset_index))}
            </div>

            <div class="{grid_class}">
                <div class="box">
                    <div class="box-title">Source Text</div>
                    {html.escape(str(source_text))}
                </div>
                <div class="box">
                    <div class="box-title">Generated Simplification</div>
                    {html.escape(str(simplified_text))}
                </div>
                {reference_box}
            </div>

            <div class="section-title">BERTScore</div>
            <div class="metric-row">
                <div class="metric">
                    <div class="metric-label">Precision</div>
                    <div class="metric-value">{precision}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Recall</div>
                    <div class="metric-value">{recall}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">F1</div>
                    <div class="metric-value">{f1}</div>
                </div>
            </div>

            <div class="section-title">Checklist Evaluation</div>
            {items_table(checklist_items, "No checklist evaluation found for this example.")}
            <div class="box">
                <div class="box-title">Overall checklist comment</div>
                {html.escape(str(overall_checklist_comment))}
            </div>

            <div class="section-title">Medical Consistency Evaluation</div>
            {items_table(medical_items, "No medical consistency evaluation found for this example.")}
            <div class="box">
                <div class="box-title">Overall medical consistency comment</div>
                {html.escape(str(overall_medical_comment))}
            </div>
        </div>
        """)

    html_parts.append("""
    </body>
    </html>
    """)

    output_path = run_dir / output_filename
    output_path.write_text("\n".join(html_parts), encoding="utf-8")

    print("Saved combined evaluation HTML:", output_path)
    return output_path

In [ ]:
# Example combined evaluation overview.
# Use this after running BERTScore and/or LLM checklist evaluation.

# generations_path = RESULTS_DIR / "medeasi_older_adults_english_results.jsonl"
# bertscore_path = generations_path.parent / "bertscore_evaluated.jsonl"
# llm_eval_path = generations_path.parent / "llm_checklist_evaluations.jsonl"
#
# overview_df = build_evaluation_overview(
#     generations_path=generations_path,
#     bertscore_path=bertscore_path if bertscore_path.exists() else None,
#     llm_eval_path=llm_eval_path if llm_eval_path.exists() else None,
# )
#
# display_evaluation_summary(overview_df)
# display_evaluation_cards(overview_df, start=0, n=5)
#
# save_evaluation_overview_html(
#     overview_df,
#     generations_path.parent / "evaluation_overview.html"
# )

In [28]:
# ============================================================
# Inspect LLM Checklist Evaluation Results
# ============================================================

llm_flat_df = flatten_llm_evaluations(llm_eval_path)

display(llm_flat_df.head())

display(llm_flat_df[[
    "profile_name",
    "checklist_mean_score",
    "checklist_applicable_items",
    "medical_consistency_mean_score",
    "medical_consistency_applicable_items",
    "overall_checklist_comment",
    "overall_medical_consistency_comment",
    "parse_error",
]].head())

,source_run_id,profile_name,source_dataset,source_dataset_index,checklist_mean_score,checklist_applicable_items,medical_consistency_mean_score,medical_consistency_applicable_items,overall_checklist_comment,overall_medical_consistency_comment,parse_error
0,7cf8e115-7dc1-4073-8a6c-e4eaf609fefe,layperson,Med-EASi,0,1.000000,9,2.0,6,The simplification is clear and uses simple la...,The simplification is medically accurate and p...,None
1,09a18de6-e2d2-495e-9606-5eae53b83f57,layperson,Med-EASi,1,1.000000,9,2.0,6,The simplification is clear and avoids jargon ...,The simplification is medically accurate and p...,None
2,70b26cc0-a0b0-45b6-9f22-53aa45af2abc,layperson,Med-EASi,2,1.900000,10,2.0,6,The simplification effectively explains medica...,"The simplification is medically accurate, pres...",None
3,8eb1660a-af76-4ac8-94fc-f31f0fa3961e,layperson,Med-EASi,3,1.900000,10,2.0,6,The simplification is well-suited for a layper...,The simplification maintains medical accuracy ...,None
4,2199098c-18f0-4e90-8681-b65f8b1f8436,layperson,Med-EASi,4,1.777778,9,2.0,6,"The simplification is clear, patient-centered,...",The simplification is medically accurate and p...,None


,profile_name,checklist_mean_score,checklist_applicable_items,medical_consistency_mean_score,medical_consistency_applicable_items,overall_checklist_comment,overall_medical_consistency_comment,parse_error
0,layperson,1.000000,9,2.0,6,The simplification is clear and uses simple la...,The simplification is medically accurate and p...,None
1,layperson,1.000000,9,2.0,6,The simplification is clear and avoids jargon ...,The simplification is medically accurate and p...,None
2,layperson,1.900000,10,2.0,6,The simplification effectively explains medica...,"The simplification is medically accurate, pres...",None
3,layperson,1.900000,10,2.0,6,The simplification is well-suited for a layper...,The simplification maintains medical accuracy ...,None
4,layperson,1.777778,9,2.0,6,"The simplification is clear, patient-centered,...",The simplification is medically accurate and p...,None


HTML RESULTS


In [13]:
# ============================================================
# Save Combined Evaluation Report as HTML
# ============================================================

from pathlib import Path
import json
import html
import pandas as pd
from collections import defaultdict


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def get_first(record, keys, default=""):
    for key in keys:
        if key in record and record[key] not in [None, ""]:
            return record[key]
    return default


def format_score(score):
    if score is None:
        return "N/A"
    return str(score)


def get_eval_root(eval_record):
    """
    Return the actual parsed evaluation object.
    Your LLM evaluation file stores it under 'evaluation_json'.
    """
    if not isinstance(eval_record, dict):
        return {}

    if isinstance(eval_record.get("evaluation_json"), dict):
        return eval_record["evaluation_json"]

    if isinstance(eval_record.get("parsed_response"), dict):
        return eval_record["parsed_response"]

    if isinstance(eval_record.get("response"), dict):
        return eval_record["response"]

    return eval_record


def extract_eval_items(eval_record, key_candidates):
    """
    Extract checklist or medical consistency items from the LLM evaluation record.
    Supports your current structure:
    evaluation_json -> checklist_scores
    evaluation_json -> medical_consistency_scores
    """
    eval_root = get_eval_root(eval_record)

    for key in key_candidates:
        if key in eval_root:
            value = eval_root[key]

            # Your current format: dict of item_id -> {score, applicable, reason}
            if isinstance(value, dict):
                items = []
                for item_id, item_data in value.items():
                    if isinstance(item_data, dict):
                        items.append({
                            "item_id": item_id,
                            "score": item_data.get("score"),
                            "applicable": item_data.get("applicable"),
                            "reason": item_data.get("reason", ""),
                        })
                return items

            # Alternative format: already a list
            if isinstance(value, list):
                return value

    return []


def get_eval_comment(eval_record, key, default=""):
    """
    Extract overall comments from nested evaluation_json.
    """
    eval_root = get_eval_root(eval_record)
    return eval_root.get(key, default)

def collect_item_score_statistics(llm_eval_records, score_key):
    """
    Collect mean scores per checklist/medical item across all evaluated examples.
    Expects:
    evaluation_json -> checklist_scores
    evaluation_json -> medical_consistency_scores
    """
    item_scores = defaultdict(list)

    for record in llm_eval_records:
        items = extract_eval_items(record, [score_key])

        for item in items:
            item_id = item.get("item_id")
            score = item.get("score")
            applicable = item.get("applicable")

            # Only include numeric scores that are applicable.
            # N/A items are excluded from the mean.
            if item_id and isinstance(score, (int, float)) and applicable is not False:
                item_scores[item_id].append(float(score))

    rows = []
    for item_id, scores in sorted(item_scores.items()):
        if scores:
            rows.append({
                "item_id": item_id,
                "mean_score": sum(scores) / len(scores),
                "count": len(scores),
            })

    return rows


def overall_mean_from_item_stats(item_stats):
    """
    Calculate overall mean from all item means, weighted by number of scores.
    """
    total_score = 0
    total_count = 0

    for row in item_stats:
        total_score += row["mean_score"] * row["count"]
        total_count += row["count"]

    if total_count == 0:
        return None

    return total_score / total_count


def item_stats_table_html(item_stats, title):
    """
    Create an HTML table for mean scores per evaluation item.
    """
    if not item_stats:
        return f"""
        <div class="section-title">{html.escape(title)}</div>
        <div class="missing">No scores available.</div>
        """

    rows = ""
    for row in item_stats:
        rows += f"""
        <tr>
            <td>{html.escape(str(row["item_id"]))}</td>
            <td class="score">{round(row["mean_score"], 3)}</td>
            <td>{row["count"]}</td>
        </tr>
        """

    return f"""
    <div class="section-title">{html.escape(title)}</div>
    <table>
        <thead>
            <tr>
                <th>Item</th>
                <th>Mean score</th>
                <th>Applicable evaluations</th>
            </tr>
        </thead>
        <tbody>
            {rows}
        </tbody>
    </table>
    """

def save_combined_evaluation_html(
    run_dir,
    generations_filename="generations.jsonl",
    bertscore_filename="bertscore_evaluated.jsonl",
    llm_eval_filename="llm_checklist_evaluations.jsonl",
    output_filename="evaluation_overview.html",
):
    run_dir = Path(run_dir)

    generations = load_jsonl(run_dir / generations_filename)
    bertscore_records = load_jsonl(run_dir / bertscore_filename)
    llm_eval_records = load_jsonl(run_dir / llm_eval_filename)

    if not generations:
        raise ValueError(f"No generation records found in {run_dir / generations_filename}")

    # Match records by order. This works if BERTScore and LLM evaluation were run on the same file/order.
    max_len = len(generations)

    html_parts = ["""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Talking Medicine Evaluation Overview</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 32px;
                background: #f5f5f5;
                color: #222;
                line-height: 1.45;
            }
            h1 {
                margin-bottom: 4px;
            }
            .subtitle {
                color: #666;
                margin-bottom: 24px;
            }
            .summary {
                background: white;
                border: 1px solid #ddd;
                border-radius: 12px;
                padding: 16px;
                margin-bottom: 24px;
            }
            .card {
                background: #fafafa;
                border: 1px solid #ddd;
                border-radius: 14px;
                padding: 18px;
                margin: 22px 0;
            }
            .meta {
                color: #666;
                font-size: 13px;
                margin-bottom: 12px;
            }
            .grid-2 {
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 14px;
                margin-bottom: 14px;
            }
            .grid-3 {
                display: grid;
                grid-template-columns: 1fr 1fr 1fr;
                gap: 14px;
                margin-bottom: 14px;
            }
            .box {
                background: white;
                border: 1px solid #e3e3e3;
                border-radius: 10px;
                padding: 12px;
                white-space: pre-wrap;
            }
            .box-title {
                font-weight: bold;
                margin-bottom: 8px;
            }
            .metric-row {
                display: flex;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 10px;
            }
            .metric {
                background: white;
                border: 1px solid #e3e3e3;
                border-radius: 10px;
                padding: 10px 12px;
                min-width: 120px;
            }
            .metric-label {
                font-size: 12px;
                color: #666;
            }
            .metric-value {
                font-size: 18px;
                font-weight: bold;
                margin-top: 4px;
            }
            table {
                width: 100%;
                border-collapse: collapse;
                background: white;
                margin-top: 8px;
                margin-bottom: 14px;
            }
            th, td {
                border: 1px solid #e3e3e3;
                padding: 8px;
                vertical-align: top;
                font-size: 13px;
            }
            th {
                background: #f0f0f0;
                text-align: left;
            }
            .score {
                font-weight: bold;
                text-align: center;
                width: 60px;
            }
            .section-title {
                font-weight: bold;
                margin-top: 16px;
                margin-bottom: 8px;
                font-size: 16px;
            }
        </style>
    </head>
    <body>
        <h1>Talking Medicine Evaluation Overview</h1>
        <div class="subtitle">
            Combined view of source texts, generated simplifications, BERTScore values, checklist evaluation, and medical consistency evaluation.
        </div>
    """]

    # Summary
    # bert_f1_values = []
    # checklist_scores = []
    # medical_scores = []

    # for i in range(max_len):
    #     if i < len(bertscore_records):
    #         f1 = get_first(bertscore_records[i], ["bertscore_f1"], None)
    #         if isinstance(f1, (int, float)):
    #             bert_f1_values.append(f1)

    #     if i < len(llm_eval_records):
    #         eval_record = llm_eval_records[i]
    #         checklist_items = extract_eval_items(eval_record, ["checklist_evaluation", "checklist_items", "checklist"])
    #         medical_items = extract_eval_items(eval_record, ["medical_consistency_evaluation", "medical_consistency_items", "medical_consistency"])

    #         for item in checklist_items:
    #             score = item.get("score")
    #             if isinstance(score, (int, float)):
    #                 checklist_scores.append(score)

    #         for item in medical_items:
    #             score = item.get("score")
    #             if isinstance(score, (int, float)):
    #                 medical_scores.append(score)

    # def mean_or_na(values):
    #     return round(sum(values) / len(values), 3) if values else "N/A"

    bert_precision_values = []
    bert_recall_values = []
    bert_f1_values = []

    checklist_item_stats = collect_item_score_statistics(
        llm_eval_records,
        "checklist_scores"
    )

    medical_item_stats = collect_item_score_statistics(
        llm_eval_records,
        "medical_consistency_scores"
    )

    overall_checklist_mean = overall_mean_from_item_stats(checklist_item_stats)
    overall_medical_mean = overall_mean_from_item_stats(medical_item_stats)

    for i, gen in enumerate(generations):
        if i < len(bertscore_records):
            precision = get_first(bertscore_records[i], ["bertscore_precision"], None)
            recall = get_first(bertscore_records[i], ["bertscore_recall"], None)
            f1 = get_first(bertscore_records[i], ["bertscore_f1"], None)

            if isinstance(precision, (int, float)):
                bert_precision_values.append(precision)

            if isinstance(recall, (int, float)):
                bert_recall_values.append(recall)

            if isinstance(f1, (int, float)):
                bert_f1_values.append(f1)


    def mean_or_na(values):
        return round(sum(values) / len(values), 3) if values else "N/A"


    def value_or_na(value):
        return round(value, 3) if isinstance(value, (int, float)) else "N/A"

    html_parts.append(f"""
        <div class="summary">
            <div class="section-title">Run Summary</div>
            <div class="metric-row">
                <div class="metric">
                    <div class="metric-label">Generated examples</div>
                    <div class="metric-value">{len(generations)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">BERTScore records</div>
                    <div class="metric-value">{len(bertscore_records)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">LLM evaluation records</div>
                    <div class="metric-value">{len(llm_eval_records)}</div>
                </div>
                <<div class="metric">
                    <div class="metric-label">Mean BERTScore Precision</div>
                    <div class="metric-value">{mean_or_na(bert_precision_values)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Mean BERTScore Recall</div>
                    <div class="metric-value">{mean_or_na(bert_recall_values)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Mean BERTScore F1</div>
                    <div class="metric-value">{mean_or_na(bert_f1_values)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Overall mean checklist score</div>
                    <div class="metric-value">{value_or_na(overall_checklist_mean)}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Overall mean medical consistency score</div>
                    <div class="metric-value">{value_or_na(overall_medical_mean)}</div>
                </div>
            </div>

            {item_stats_table_html(checklist_item_stats, "Mean Scores per Checklist Question")}

            {item_stats_table_html(medical_item_stats, "Mean Scores per Medical Consistency Question")}
        </div>
    """)

    for i, gen in enumerate(generations):
        bert = bertscore_records[i] if i < len(bertscore_records) else {}
        llm_eval = llm_eval_records[i] if i < len(llm_eval_records) else {}

        source_text = get_first(gen, ["source_text", "original_text", "expert_text", "Expert", "input_text", "text"], "[No source text found]")
        simplified_text = get_first(gen, ["simplified_text", "generated_text", "output_text", "simplification", "model_output"], "[No simplification found]")
        reference_text = get_first(gen, ["reference_text", "simple_text", "Simple", "reference", "gold_text"], "")

        profile = get_first(gen, ["profile_name", "profile", "patient_profile"], "N/A")
        target_language = get_first(gen, ["target_language", "language"], "N/A")
        dataset_index = get_first(gen, ["dataset_index", "index", "text_id", "id"], str(i))

        precision = get_first(bert, ["bertscore_precision"], "N/A")
        recall = get_first(bert, ["bertscore_recall"], "N/A")
        f1 = get_first(bert, ["bertscore_f1"], "N/A")

        if isinstance(precision, float):
            precision = round(precision, 3)
        if isinstance(recall, float):
            recall = round(recall, 3)
        if isinstance(f1, float):
            f1 = round(f1, 3)

        checklist_items = extract_eval_items(
            llm_eval,
            ["checklist_scores", "checklist_evaluation", "checklist_items", "checklist"]
        )

        medical_items = extract_eval_items(
            llm_eval,
            ["medical_consistency_scores", "medical_consistency_evaluation", "medical_consistency_items", "medical_consistency"]
        )

        overall_checklist_comment = get_eval_comment(
            llm_eval,
            "overall_checklist_comment",
            ""
        )

        overall_medical_comment = get_eval_comment(
            llm_eval,
            "overall_medical_consistency_comment",
            ""
        )

        reference_box = ""
        grid_class = "grid-2"
        if reference_text:
            grid_class = "grid-3"
            reference_box = f"""
            <div class="box">
                <div class="box-title">Reference Simplification</div>
                {html.escape(str(reference_text))}
            </div>
            """

        def items_table(items, empty_message):
            if not items:
                return f"<p>{html.escape(empty_message)}</p>"

            rows = ""
            for item in items:
                item_id = html.escape(str(item.get("item_id", item.get("id", ""))))
                question = html.escape(str(item.get("question", "")))
                score = html.escape(format_score(item.get("score")))
                reason = html.escape(str(item.get("reason", item.get("justification", ""))))

                rows += f"""
                <tr>
                    <td>{item_id}</td>
                    <td>{question}</td>
                    <td class="score">{score}</td>
                    <td>{reason}</td>
                </tr>
                """

            return f"""
            <table>
                <thead>
                    <tr>
                        <th>Item</th>
                        <th>Question</th>
                        <th>Score</th>
                        <th>Reason</th>
                    </tr>
                </thead>
                <tbody>
                    {rows}
                </tbody>
            </table>
            """

        html_parts.append(f"""
        <div class="card">
            <div class="meta">
                <b>Example {i}</b> |
                Profile: {html.escape(str(profile))} |
                Language: {html.escape(str(target_language))} |
                ID/Index: {html.escape(str(dataset_index))}
            </div>

            <div class="{grid_class}">
                <div class="box">
                    <div class="box-title">Source Text</div>
                    {html.escape(str(source_text))}
                </div>
                <div class="box">
                    <div class="box-title">Generated Simplification</div>
                    {html.escape(str(simplified_text))}
                </div>
                {reference_box}
            </div>

            <div class="section-title">BERTScore</div>
            <div class="metric-row">
                <div class="metric">
                    <div class="metric-label">Precision</div>
                    <div class="metric-value">{precision}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">Recall</div>
                    <div class="metric-value">{recall}</div>
                </div>
                <div class="metric">
                    <div class="metric-label">F1</div>
                    <div class="metric-value">{f1}</div>
                </div>
            </div>

            <div class="section-title">Checklist Evaluation</div>
            {items_table(checklist_items, "No checklist evaluation found for this example.")}
            <div class="box">
                <div class="box-title">Overall checklist comment</div>
                {html.escape(str(overall_checklist_comment))}
            </div>

            <div class="section-title">Medical Consistency Evaluation</div>
            {items_table(medical_items, "No medical consistency evaluation found for this example.")}
            <div class="box">
                <div class="box-title">Overall medical consistency comment</div>
                {html.escape(str(overall_medical_comment))}
            </div>
        </div>
        """)

    html_parts.append("""
    </body>
    </html>
    """)

    output_path = run_dir / output_filename
    output_path.write_text("\n".join(html_parts), encoding="utf-8")

    print("Saved combined evaluation HTML:", output_path)
    return output_path

In [3]:
# Use manual folder
run_dir = Path("./outputs/runs/layperson")

html_path = save_combined_evaluation_html(
    run_dir=run_dir,
    generations_filename="generations.jsonl",
    bertscore_filename="bertscore_evaluated.jsonl",
    llm_eval_filename="llm_checklist_evaluations.jsonl",
    output_filename="evaluation_overview.html",
)

print(html_path)

NameError: name 'extract_score_items' is not defined

In [15]:
from pathlib import Path

run_dir = Path("./outputs/runs/cognitive_disabilities")

html_path = save_combined_evaluation_html(
    run_dir=run_dir,
    generations_filename="generations.jsonl",
    bertscore_filename="bertscore_evaluated.jsonl",
    llm_eval_filename="llm_checklist_evaluations.jsonl",
    output_filename="evaluation_overview.html",
)

print(html_path)

Saved combined evaluation HTML: outputs\runs\cognitive_disabilities\evaluation_overview.html
outputs\runs\cognitive_disabilities\evaluation_overview.html


## 11. Notes for Thesis Reporting

Suggested implementation description:

- The implementation uses reusable patient-profile prompt designs.
- These profile designs do not specify the output language and do not explicitly list complexity-dimension labels.
- A shared prompt wrapper adds the selected output language, the source text, medical correctness constraints, and output requirements.
- Manual input and Med-EASi processing use the same prompt builder and GPT-4o generation function.
- Outputs are saved with metadata to support reproducibility and later evaluation.
- The evaluation module combines BERTScore-based semantic similarity analysis with LLM-based checklist evaluation.
- BERTScore is interpreted as a supporting semantic-preservation signal, while factual consistency should be interpreted together with qualitative or expert evaluation.

In [ ]:
# Final configuration check. This cell deliberately does not print the API key.
print("Endpoint:", AZURE_OPENAI_ENDPOINT)
print("API version:", AZURE_OPENAI_API_VERSION)
print("Deployment name:", AZURE_OPENAI_CHAT_DEPLOYMENT_NAME)
print("API key loaded:", bool(AZURE_OPENAI_API_KEY) and AZURE_OPENAI_API_KEY != "PASTE_YOUR_REGENERATED_KEY_HERE")